# OpenArm 2.0 Phase 2 — Google Drive Version

This notebook is rewritten to use this persistent Google Drive path:

```text
/content/drive/MyDrive/Colab Notebooks
```

It stores the project repo, input zips, generated outputs, and final submission zip in Google Drive instead of temporary `/content`.

Expected files in Google Drive:

```text
/content/drive/MyDrive/Colab Notebooks/openarm-data-pipeline-phase2-colab.zip
/content/drive/MyDrive/Colab Notebooks/openarm_phase1_outputs.zip
```

The second file is optional, but useful if you want to compare Phase 1 and Phase 2 outputs.


## 1. Mount Google Drive and define paths


In [ ]:
from pathlib import Path
import os, sys, json, zipfile, shutil, subprocess, textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")

DRIVE_WORKDIR = Path("/content/drive/MyDrive/Colab Notebooks")
DRIVE_WORKDIR.mkdir(parents=True, exist_ok=True)

PROJECT_ZIP = DRIVE_WORKDIR / "openarm-data-pipeline-phase2-colab.zip"
PHASE1_ZIP = DRIVE_WORKDIR / "openarm_phase1_outputs.zip"
PROJECT_DIR = DRIVE_WORKDIR / "openarm-data-pipeline"

print("Drive workdir:", DRIVE_WORKDIR)
print("Project zip:", PROJECT_ZIP, "exists:", PROJECT_ZIP.exists())
print("Phase 1 zip:", PHASE1_ZIP, "exists:", PHASE1_ZIP.exists())
print("Project dir:", PROJECT_DIR, "exists:", PROJECT_DIR.exists())


## 2. Upload zips into Google Drive if missing

Run this cell if the files are not already in:

```text
/content/drive/MyDrive/Colab Notebooks
```


In [ ]:
from google.colab import files

missing = []
if not PROJECT_ZIP.exists():
    missing.append(PROJECT_ZIP.name)
if not PHASE1_ZIP.exists():
    missing.append(PHASE1_ZIP.name + " (optional)")

if missing:
    print("Missing files:", missing)
    print("Upload openarm-data-pipeline-phase2-colab.zip and optionally openarm_phase1_outputs.zip")
    uploaded = files.upload()

    for name in uploaded.keys():
        src = Path(name)
        dst = DRIVE_WORKDIR / name
        if src.exists():
            shutil.move(str(src), str(dst))
            print("Moved to Drive:", dst)
else:
    print("Required zip files already exist in Google Drive.")


## 3. Unzip Phase 2 project into Google Drive


In [ ]:
if not PROJECT_ZIP.exists():
    raise FileNotFoundError(
        f"Missing {PROJECT_ZIP}. Upload openarm-data-pipeline-phase2-colab.zip first."
    )

if PROJECT_DIR.exists():
    print("Removing existing project directory:", PROJECT_DIR)
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(PROJECT_ZIP, "r") as zf:
    zf.extractall(DRIVE_WORKDIR)

if not PROJECT_DIR.exists():
    # Some zip tools may extract an extra nested folder. Try to locate it.
    matches = list(DRIVE_WORKDIR.rglob("openarm-data-pipeline"))
    print("Found matches:", matches)
    if matches:
        PROJECT_DIR = matches[0]
    else:
        raise FileNotFoundError("Could not find extracted openarm-data-pipeline directory.")

print("Project extracted to:", PROJECT_DIR)
print("Top-level files:")
for p in sorted(PROJECT_DIR.iterdir()):
    print(" ", p.name)


## 4. Import Phase 1 outputs into the Drive project


In [ ]:
phase1_out = PROJECT_DIR / "outputs" / "phase1"
phase1_out.mkdir(parents=True, exist_ok=True)

if PHASE1_ZIP.exists():
    tmp_phase1 = DRIVE_WORKDIR / "_phase1_unzipped_tmp"
    if tmp_phase1.exists():
        shutil.rmtree(tmp_phase1)
    tmp_phase1.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(PHASE1_ZIP, "r") as zf:
        zf.extractall(tmp_phase1)

    candidates = list(tmp_phase1.rglob("phase1_audit_summary.csv"))
    if candidates:
        src_dir = candidates[0].parent
        for f in src_dir.iterdir():
            if f.is_file():
                shutil.copy(f, phase1_out / f.name)
        print("Copied Phase 1 outputs from:", src_dir)
    else:
        print("No phase1_audit_summary.csv found inside Phase 1 zip.")

    shutil.rmtree(tmp_phase1)
else:
    print("No Phase 1 zip found. Skipping Phase 1 import.")

print("Phase 1 output directory:", phase1_out)
print("Files:", [p.name for p in phase1_out.iterdir()])


## 5. Review Phase 1 result summary, if available


In [ ]:
summary_path = phase1_out / "phase1_audit_summary.csv"

if summary_path.exists():
    phase1_df = pd.read_csv(summary_path)
    display(phase1_df.head())
    display(phase1_df.describe(include="all"))

    print("Kept candidates:", int(phase1_df["keep_candidate"].sum()), "/", len(phase1_df))
    print("Mean episode length:", phase1_df["num_steps"].mean())
    print(
        "Max joint velocity range:",
        phase1_df["max_abs_joint_velocity"].min(),
        "to",
        phase1_df["max_abs_joint_velocity"].max(),
    )
    print("Mean bad frame ratio:", phase1_df["bad_frame_ratio"].mean())
else:
    print("No previous Phase 1 summary found.")


## 6. Install requirements from the Drive project


In [ ]:
os.chdir(PROJECT_DIR)
print("Current directory:", Path.cwd())

req_path = PROJECT_DIR / "requirements.txt"
if not req_path.exists():
    req_path = PROJECT_DIR / "requirement.txt"

print("Installing from:", req_path)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_path)])
print("Installed requirements.")


## 7. Inspect the selected LeRobot dataset


In [ ]:
os.chdir(PROJECT_DIR)
print("Current directory:", Path.cwd())

subprocess.check_call([
    sys.executable,
    "scripts/inspect_dataset.py",
    "--repo-id",
    "lerobot/libero_object_image",
])


## 8. Write Colab config safely

This replaces the previous `%%writefile configs/curation_colab.yaml` cell.

Using `Path.write_text()` avoids the `FileNotFoundError` caused by missing `configs/` or incorrect working directory.


In [ ]:
os.chdir(PROJECT_DIR)

config_dir = PROJECT_DIR / "configs"
config_dir.mkdir(parents=True, exist_ok=True)

config_text = '''repo_id: "lerobot/libero_object_image"

state_key: null
action_key: null
timestamp_key: null
wrist_image_key: "observation.images.wrist_image"

# Start smaller in Colab. Change to null for full dataset.
max_episodes: 50

# Teleoperation filters
min_episode_len: 20
max_joint_velocity: 20.0
max_timestamp_gap_ratio: 3.0

# Wrist / egocentric video filters
# These are starter thresholds. Use score plots to calibrate.
blur_threshold: 30.0
exposure_threshold: 0.50
frozen_frame_threshold: 0.5
max_bad_frame_ratio: 0.25
video_frame_stride: 5

# Curation
smooth_window: 5
drop_bad_frames_with_mask: true

output_dir: "outputs"
'''

config_path = config_dir / "curation_colab.yaml"
config_path.write_text(config_text, encoding="utf-8")

print("Wrote config:", config_path)
print(config_path.read_text())


## 9. Run Phase 2 audit


In [ ]:
os.chdir(PROJECT_DIR)
print("Current directory:", Path.cwd())

subprocess.check_call([
    sys.executable,
    "scripts/audit_dataset.py",
    "--config",
    "configs/curation_colab.yaml",
])


## 10. Load and interpret audit outputs


In [ ]:
output_dir = PROJECT_DIR / "outputs"
audit_summary_path = output_dir / "audit_summary.csv"
audit_metadata_path = output_dir / "audit_metadata.json"

audit_df = pd.read_csv(audit_summary_path)
display(audit_df.head())
display(audit_df.describe(include="all"))

with open(audit_metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Candidate episodes kept by audit:", int(audit_df["keep_candidate"].sum()), "/", len(audit_df))
print("\nUnique filter reasons:")
display(audit_df["filter_reasons"].value_counts(dropna=False))

print("\nMetadata preview:")
print(json.dumps(metadata, indent=2)[:3000])


## 11. Show audit plots from Drive outputs


In [ ]:
fig_dir = PROJECT_DIR / "outputs" / "figures"

if not fig_dir.exists():
    print("No figure directory found:", fig_dir)
else:
    for path in sorted(fig_dir.glob("*.png")):
        print(path.name)
        img = plt.imread(path)
        plt.figure(figsize=(8, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.show()


## 12. Diagnose video rejection reasons


In [ ]:
from collections import Counter

report_path = PROJECT_DIR / "outputs" / "audit_report.json"

with open(report_path, "r", encoding="utf-8") as f:
    reports = json.load(f)

reason_counts = Counter()
blur_scores, exposure_scores, frame_diffs = [], [], []

for r in reports:
    for fr in r["egocentric_video_report"]["frame_reports"]:
        for reason in fr["reasons"]:
            reason_counts[reason] += 1
        blur_scores.append(fr["blur_score"])
        exposure_scores.append(fr["exposure_score"])
        if fr["frame_difference"] is not None:
            frame_diffs.append(fr["frame_difference"])

print("Video rejection reason counts:", reason_counts)

def stats(name, values):
    if not values:
        print(name, "no values")
        return
    arr = np.asarray(values)
    print(
        f"{name}: min={arr.min():.4f}, mean={arr.mean():.4f}, "
        f"median={np.median(arr):.4f}, max={arr.max():.4f}"
    )

stats("Blur", blur_scores)
stats("Exposure", exposure_scores)
stats("Frame diff", frame_diffs)


## 13. Visualize wrist-camera frames


In [ ]:
os.chdir(PROJECT_DIR)

subprocess.check_call([
    sys.executable,
    "scripts/visualize_episode.py",
    "--config",
    "configs/curation_colab.yaml",
    "--episode-id",
    "0",
    "--num-frames",
    "6",
])

vis_paths = sorted((PROJECT_DIR / "outputs" / "figures").glob("*wrist_frames.png"))
for path in vis_paths[-3:]:
    print(path.name)
    img = plt.imread(path)
    plt.figure(figsize=(12, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


## 14. Run curation


In [ ]:
os.chdir(PROJECT_DIR)

subprocess.check_call([
    sys.executable,
    "scripts/curate_dataset.py",
    "--config",
    "configs/curation_colab.yaml",
])


## 15. Inspect curated manifest and arrays


In [ ]:
manifest_path = PROJECT_DIR / "outputs" / "curated_manifest.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest_df = pd.DataFrame([
    {
        "episode_id": m["episode_id"],
        "kept": m["kept"],
        "drop_reasons": ", ".join(m["drop_reasons"]),
        "num_original_steps": m["num_original_steps"],
        "num_valid_aligned_steps": m["num_valid_aligned_steps"],
        "array_path": m.get("array_path", ""),
    }
    for m in manifest
])

display(manifest_df.head())
display(manifest_df["kept"].value_counts())

curated_dir = PROJECT_DIR / "outputs" / "curated_arrays"
curated_files = sorted(curated_dir.glob("*.npz"))
print("Curated array files:", len(curated_files))

if curated_files:
    sample_npz = curated_files[0]
    data = np.load(sample_npz)
    print("\nSample curated file:", sample_npz)
    print("Keys:", data.files)
    for k in data.files:
        print(k, data[k].shape, data[k].dtype)

    states = data["states"]
    states_smooth = data["states_smooth"]

    plt.figure(figsize=(8, 4))
    dim = 0
    plt.plot(states[:, dim], label="raw state dim 0")
    plt.plot(states_smooth[:, dim], label="smoothed state dim 0")
    plt.title("Raw vs Smoothed State Trajectory")
    plt.xlabel("Curated timestep")
    plt.ylabel("State value")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No curated arrays found. Check thresholds or audit filter reasons.")


## 16. Export Label Studio task stubs


In [ ]:
os.chdir(PROJECT_DIR)

subprocess.check_call([
    sys.executable,
    "scripts/export_label_studio_tasks.py",
    "--config",
    "configs/curation_colab.yaml",
    "--manifest",
    "outputs/curated_manifest.json",
    "--out",
    "outputs/label_studio_tasks.json",
])

tasks_path = PROJECT_DIR / "outputs" / "label_studio_tasks.json"
with open(tasks_path, "r", encoding="utf-8") as f:
    tasks = json.load(f)

print("Number of task stubs:", len(tasks))
print(json.dumps(tasks[:3], indent=2))


## 17. Generate Phase 2 results write-up


In [ ]:
result_md = PROJECT_DIR / "docs" / "RESULTS_PHASE2.md"
result_md.parent.mkdir(parents=True, exist_ok=True)

num_audited = len(audit_df)
num_keep = int(audit_df["keep_candidate"].sum())
mean_len = float(audit_df["num_steps"].mean())
min_len = int(audit_df["num_steps"].min())
max_len = int(audit_df["num_steps"].max())
max_vel_min = float(audit_df["max_abs_joint_velocity"].min())
max_vel_max = float(audit_df["max_abs_joint_velocity"].max())
mean_bad = float(audit_df["bad_frame_ratio"].mean()) if audit_df["bad_frame_ratio"].notna().any() else None
filter_counts = audit_df["filter_reasons"].value_counts(dropna=False).to_dict()

content = f'''# Phase 2 Results

## Dataset

- Dataset: `lerobot/libero_object_image`
- Audited episodes: {num_audited}
- Selected wrist/egocentric stream: `{metadata["keys"]["wrist_image_key"]}`
- State key: `{metadata["keys"]["state_key"]}`
- Action key: `{metadata["keys"]["action_key"]}`
- Timestamp key: `{metadata["keys"]["timestamp_key"]}`

## Teleoperation audit

The audited episodes have trajectory lengths between {min_len} and {max_len} frames, with an average length of {mean_len:.2f} frames.

The maximum absolute joint/state velocity across episodes ranged from {max_vel_min:.4f} to {max_vel_max:.4f} under the current finite-difference estimate.

The audit checks for:

- NaN/Inf state values
- per-dimension state min/max/mean/std
- timestamp gaps
- finite-difference velocity spikes

## Egocentric / wrist-camera audit

The wrist-camera audit sampled frames with stride `{metadata["thresholds"]["video_frame_stride"]}` and computed:

- blur score using variance of Laplacian
- exposure saturation ratio
- adjacent-frame difference for frozen-frame detection

Mean bad-frame ratio: {mean_bad}

Video rejection reason counts:

```text
{metadata.get("video_reason_counts", {})}
```

## Curation result

Candidate episodes kept by the current filters: {num_keep} / {num_audited}

Filter reason counts:

```text
{filter_counts}
```

The curation step saves a manifest with `original_indices` and `valid_alignment_mask`, so downstream training can preserve the mapping between robot state, action, timestamp, and wrist-camera frame.

## Interpretation

Teleoperation filtering is mostly numerical and kinematic: short episodes, invalid states, timestamp gaps, and unrealistic velocity spikes.

Egocentric filtering is perceptual and threshold-sensitive. Blur/exposure/frozen-frame thresholds should be calibrated using score distributions and manual visualization before dropping full episodes.

## Next steps

- run the audit on all episodes by setting `max_episodes: null`
- manually inspect frames near the blur/exposure thresholds
- export real video clips for Label Studio rather than task stubs
- add object visibility and task-success labels
- train a small wrist-camera success detector
'''

result_md.write_text(content, encoding="utf-8")
print("Wrote:", result_md)
print(result_md.read_text())


## 18. Zip final Phase 2 repo back into Google Drive


In [ ]:
FINAL_ZIP = DRIVE_WORKDIR / "openarm-data-pipeline-final-colab-drive.zip"

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

shutil.make_archive(
    str(FINAL_ZIP).replace(".zip", ""),
    "zip",
    root_dir=str(DRIVE_WORKDIR),
    base_dir="openarm-data-pipeline",
)

print("Created:", FINAL_ZIP)
print("Size MB:", FINAL_ZIP.stat().st_size / 1e6)


## 19. Optional download

The final zip is already saved in Google Drive:

```text
/content/drive/MyDrive/Colab Notebooks/openarm-data-pipeline-final-colab-drive.zip
```

Run the cell below only if you also want to download it to your local machine.


In [ ]:
from google.colab import files

FINAL_ZIP = DRIVE_WORKDIR / "openarm-data-pipeline-final-colab-drive.zip"
if FINAL_ZIP.exists():
    files.download(str(FINAL_ZIP))
else:
    print("Final zip not found yet. Run the previous cell first.")
